# STHAN Native Adapter v1

目标：把 `sthan_mse` 从 spatio-temporal attention feature proxy 推进到 `src/model` 的 native STHAN-style estimator。Notebook 保留实验流程、外部参考 URL、matrix sidecar 和中文结论；可复用训练、回测、analysis 逻辑都调用 `src/`。


In [1]:
from pathlib import Path
import json
import sys

REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'AGENTS.md').exists() and (path / 'src').exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data import build_supervised_dataset, load_soxl_support_frame
from src.model.capabilities import model_capability_table, model_fidelity_summary
from src.model.graph import STHANGraphRegressor
from src.model.matrix import run_quantbench_matrix_smoke
from src.model.training import fit_estimator_for_model, model_adapter_name

OUTPUT_DIR = REPO_ROOT / 'research' / 'sthan_native_adapter_v1'
CARD_DIR = OUTPUT_DIR / 'matrix_card'
MODEL_NAMES = ('ridge_mse', 'hypergraph_mse', 'sthan_mse', 'thgnn_mse', 'sthgcn_mse')

outputs = run_quantbench_matrix_smoke(
    card_dir=CARD_DIR,
    start='2026-01-01',
    end='2026-06-09',
    train_end='2026-04-30',
    model_names=MODEL_NAMES,
)

frame = load_soxl_support_frame(start='2026-01-01', end='2026-06-09')
dataset = build_supervised_dataset(frame, horizon=1)
train = dataset.frame.loc[dataset.frame['date'] <= '2026-04-30'].copy()
feature_columns = tuple(dataset.feature_columns[:8])
estimator, constant_prediction = fit_estimator_for_model('sthan_mse', train=train, feature_columns=feature_columns)
capability = model_capability_table(('sthan_mse',)).iloc[0].to_dict()
fidelity = model_fidelity_summary()

assert constant_prediction is None
assert isinstance(estimator, STHANGraphRegressor)
assert estimator.feature_count_ == 8
assert estimator.node_count_ == 8
assert estimator.temporal_channel_count_ == 4
assert estimator.hyperedge_count_ >= 4
assert estimator.incidence_matrix_.shape == (8, estimator.hyperedge_count_)
assert estimator.hypergraph_propagation_.shape == (8, 8)
assert estimator.temporal_attention_weights_.shape == (4,)
assert estimator.spatial_attention_weights_.shape == (8, 8)
assert hasattr(estimator, 'model_')
assert capability['paper_fidelity'] == 'native_graph'
assert capability['is_proxy'] is False
assert model_adapter_name('sthan_mse') == 'sthan_spatio_temporal_attention_mse_v1'
assert 'sthan_mse' in fidelity['native_models']
assert 'sthan_mse' not in fidelity['proxy_models']
assert len(fidelity['native_models']) == 26
assert len(fidelity['proxy_models']) == 6

comparison_row = outputs.model_comparison.loc[outputs.model_comparison['model'].eq('sthan_mse')].iloc[0].to_dict()
ledger_row = outputs.model_run_ledger.loc[outputs.model_run_ledger['model'].eq('sthan_mse')].iloc[0].to_dict()
summary = {
    'schema_version': 'research.sthan_native_adapter_v1',
    'status': 'completed',
    'model': 'sthan_mse',
    'adapter': model_adapter_name('sthan_mse'),
    'estimator': estimator.__class__.__name__,
    'node_count': estimator.node_count_,
    'feature_count': estimator.feature_count_,
    'temporal_channel_count': estimator.temporal_channel_count_,
    'hyperedge_count': estimator.hyperedge_count_,
    'hidden_size': estimator.hidden_size,
    'paper_fidelity': capability['paper_fidelity'],
    'is_proxy': bool(capability['is_proxy']),
    'full_matrix_native_model_count': len(fidelity['native_models']),
    'full_matrix_proxy_model_count': len(fidelity['proxy_models']),
    'matrix_card': str(CARD_DIR.relative_to(REPO_ROOT)),
    'matrix_models': sorted(outputs.predictions['model'].astype(str).unique().tolist()),
    'target_model_return_pct': float(comparison_row.get('return_pct', 0.0)),
    'target_model_sharpe': float(comparison_row.get('sharpe', 0.0)),
    'target_model_max_drawdown_pct': float(comparison_row.get('max_drawdown_pct', 0.0)),
    'ledger_trained': bool(ledger_row.get('trained', False)),
    'ledger_backtested': bool(ledger_row.get('backtested', False)),
    'external_references': [
        {
            'source_id': 'doi:10.1016/j.asoc.2024.111329',
            'url': 'https://doi.org/10.1016/j.asoc.2024.111329',
            'usage': '参考动态 hypergraph spatio-temporal stock trend prediction 的公开题录与结构方向；本仓库未复制论文全文或第三方代码。',
        },
        {
            'source_id': 'github:tanjingme/CI-STHPAN',
            'url': 'https://github.com/tanjingme/CI-STHPAN',
            'usage': '参考 channel-independent spatio-temporal hypergraph attention stock selection 的开源项目边界；本实现仅保留 compact adapter 结构。',
        },
    ],
    'artifact_links': {
        'analysis_summary': str((CARD_DIR / 'analysis_summary.json').relative_to(REPO_ROOT)),
        'model_comparison': str((CARD_DIR / 'model_comparison.csv').relative_to(REPO_ROOT)),
        'model_run_ledger': str((CARD_DIR / 'model_run_ledger.csv').relative_to(REPO_ROOT)),
        'manifest': str((CARD_DIR / 'analysis_manifest.json').relative_to(REPO_ROOT)),
        'qlib_risk_metrics': str((CARD_DIR / 'qlib_risk_metrics.csv').relative_to(REPO_ROOT)),
        'model_effect_deltas': str((CARD_DIR / 'model_effect_deltas.csv').relative_to(REPO_ROOT)),
    },
}
(OUTPUT_DIR / 'notebook_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
summary


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'schema_version': 'research.sthan_native_adapter_v1',
 'status': 'completed',
 'model': 'sthan_mse',
 'adapter': 'sthan_spatio_temporal_attention_mse_v1',
 'estimator': 'STHANGraphRegressor',
 'node_count': 8,
 'feature_count': 8,
 'temporal_channel_count': 4,
 'hyperedge_count': 11,
 'hidden_size': 8,
 'paper_fidelity': 'native_graph',
 'is_proxy': False,
 'full_matrix_native_model_count': 26,
 'full_matrix_proxy_model_count': 6,
 'matrix_card': 'research/sthan_native_adapter_v1/matrix_card',
 'matrix_models': ['hypergraph_mse',
  'ridge_mse',
  'sthan_mse',
  'sthgcn_mse',
  'thgnn_mse'],
 'target_model_return_pct': 20.01990094044881,
 'target_model_sharpe': 2.05156668118373,
 'target_model_max_drawdown_pct': 34.93762308925087,
 'ledger_trained': True,
 'ledger_backtested': True,
 'external_references': [{'source_id': 'doi:10.1016/j.asoc.2024.111329',
   'url': 'https://doi.org/10.1016/j.asoc.2024.111329',
   'usage': '参考动态 hypergraph spatio-temporal stock trend prediction 的公开题录与结构方

## 外部参考摘要

- `Stock trend prediction based on dynamic hypergraph spatio-temporal network`：公开题录显示该方向把动态 hypergraph 与时空股票趋势预测结合。参考：https://doi.org/10.1016/j.asoc.2024.111329
- `CI-STHPAN`：GitHub 项目说明其目标是 channel-independent spatio-temporal hypergraph attention stock selection。参考：https://github.com/tanjingme/CI-STHPAN
- 本仓库实现：SOXL 单资产监督任务当前没有真实股票横截面和多资产行业超边，本轮先把特征列视为 feature nodes，用四通道 temporal signal、hypergraph propagation、temporal decay attention 和 node-level spatial attention 形成 compact native adapter。后续若接入股票池/行业/ETF 成分关系，应替换 hyperedge provider 和 objective，而不是把真实关系写死在 estimator 内。


## 中文结论

- 做了什么：在 `src/model.graph` 增加 `STHANGraphRegressor`，把 `sthan_mse` 从 `GraphFeatureRidgeRegressor(graph_kind='sthan')` proxy 升级为 `native_graph`。实现使用四通道 temporal node signal、temporal decay attention、feature-node hypergraph propagation 和 node-level spatial attention。
- 核心指标：本 notebook 训练并回测 `ridge_mse`、`hypergraph_mse`、`sthan_mse`、`thgnn_mse`、`sthgcn_mse`，输出 `model_comparison.csv`、`daily_backtest.csv`、`model_run_ledger.csv`、`qlib_risk_metrics.csv`、`model_effect_deltas.csv`、`model_regime_window_diagnostics.csv` 和 manifest。
- 模型实现状态：全模型注册表当前 native=26、proxy=6；`sthan_mse` 已进入 `native_graph`，adapter 仍为 `sthan_spatio_temporal_attention_mse_v1`。
- Regime/区间差异：本轮继续复用 `src/analysis` 的 window/regime/qlib 风险表，具体 winner/loser 以 `matrix_card/analysis_summary.json`、`model_review_briefing.csv` 和 `model_effect_deltas.csv` 为准。
- 是否晋升：否。本次是模型实现和链路覆盖升级，不直接晋升生产策略。
- 风险：当前 feature-node hyperedge 不是股票关系图；STHAN-style compact adapter 不能等价于完整多资产论文实现，真实横截面股票池、行业/ETF 成分、资金流关系和更长 walk-forward 仍需验证。
- 下一步：刷新全模型 ledger 和 market-regime analysis，使 native/proxy 计数、adapter fidelity 和模型回测记录同步到最新框架。
